In [9]:
import pandas as pd,numpy as np
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset,DataLoader
import torch

In [10]:
model_name = "distilbert-base-uncased"
LR =  2e-5

In [11]:
data = pd.read_csv('/content/open_multitask_nlp_v1_train.csv')

In [12]:
sentiment_map = {
    'negative': 0,
    'neutral': 1,
    'positive': 2
}
behavior_map = {
    'very_poor': 0,
    'poor': 1,
    'below_average': 2,
    'neutral': 3,
    'slightly_positive': 4,
    'good': 5,
    'very_good': 6,
    'excellent': 7
}

category_map = {
    'account_issue': 0,
    'complaint': 1,
    'feedback': 2,
    'need_help': 3,
    'technical_issue': 4,
    'feature_request': 5,
    'billing_payment': 6,
    'policy_question': 7,
    'service_delay': 8,
    'general_inquiry': 9,
    'praise_appreciation': 10,
    'customer_opinion': 11
}
urgency_map = {
    'low': 0,
    'medium': 1,
    'high': 2,
    'critical': 3
}


In [13]:
text = data['text'].tolist()
sentiment = data['sentiment'].tolist()
behavior = data['behavior_score'].tolist()
category = data['category_1'].tolist()
urgency = data['urgency'].tolist()

In [14]:
sentiment = [sentiment_map[x] for x in sentiment]
behavior  = [behavior_map[x] for x in behavior]
category  = [category_map[x] for x in category]
urgency   = [urgency_map[x] for x in urgency]


In [15]:
from sklearn.model_selection import train_test_split
X_train , X_val, senti_train,senti_val, behavior_train,behavior_val,cat_train,cat_val ,urgency_train,urgency_val =train_test_split(text,sentiment,behavior,category,urgency,test_size=0.33,random_state=11)


In [16]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [27]:

max_len = 250

In [28]:
max_len

250

In [19]:
class MultiTaskDataset(Dataset):
    def __init__(self,texts,sentiment,behavior,category,urgency,tokenizer,max_len):
        self.texts=texts
        self.sentiments = sentiment
        self.behavior = behavior
        self.category = category
        self.urgency = urgency
        self.tokenizer = tokenizer
        self.max_len= max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index],
            truncation = True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            "input_ids":encoding['input_ids'].squeeze(0),
            "attention_mask":encoding["attention_mask"].squeeze(0),
            "sentiment_labels":torch.tensor(self.sentiments[index],dtype=torch.long),
            "behavior_labels":torch.tensor(self.behavior[index],dtype=torch.long),
            "category_labels":torch.tensor(self.category[index],dtype=torch.long),
            "urgency_labels":torch.tensor(self.urgency[index],dtype=torch.long)
        }

In [20]:
train_dataset = MultiTaskDataset(texts=X_train,
                                sentiment=senti_train,
                                behavior=behavior_train,
                                category=cat_train,
                                urgency=urgency_train,
                                tokenizer=tokenizer,
                                max_len=max_len
                                )
val_dataset = MultiTaskDataset(texts=X_val,
                                sentiment=senti_val,
                                behavior=behavior_val,
                                category=cat_val,
                                urgency=urgency_val,
                                tokenizer=tokenizer,
                                max_len=max_len
                                )

In [21]:
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=64)

In [29]:
class MultiTaskeBert(torch.nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size

        self.sentiment = torch.nn.Linear(hidden, 3)
        self.behavior  = torch.nn.Linear(hidden, 8)
        self.category  = torch.nn.Linear(hidden, 12)
        self.urgency   = torch.nn.Linear(hidden, 4)

        self.loss_fn = torch.nn.CrossEntropyLoss()

    def forward(
        self,
        input_ids,
        attention_mask,
        sentiment_labels=None,
        behavior_labels=None,
        category_labels=None,
        urgency_labels=None
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:, 0]

        s_logits = self.sentiment(cls)
        b_logits = self.behavior(cls)
        c_logits = self.category(cls)
        u_logits = self.urgency(cls)   # ✅ FIXED

        # always return logits
        output = {
            "sentiment_logits": s_logits,
            "behavior_logits": b_logits,
            "category_logits": c_logits,
            "urgency_logits": u_logits
        }

        # compute loss only if labels are provided
        if sentiment_labels is not None:
            loss = (
                self.loss_fn(s_logits, sentiment_labels) +
                self.loss_fn(b_logits, behavior_labels) +
                self.loss_fn(c_logits, category_labels) +
                self.loss_fn(u_logits, urgency_labels)
            )
            output["loss"] = loss

        return output


In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [31]:
model = MultiTaskeBert(model_name).to(device)
optimizer = torch.optim.AdamW(model.parameters(),lr=LR)

In [36]:
Epochs = 10

In [ ]:
for epochs in range(Epochs):
    print(f"Epoch {epochs+1}/{Epochs}")
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch["attention_mask"].to(device)
        s_labels = batch["sentiment_labels"].to(device)
        b_labels = batch["behavior_labels"].to(device)
        c_labels = batch["category_labels"].to(device)
        u_labels = batch["urgency_labels"].to(device)

        outputs = model(
            input_ids,
            attention_mask,
            s_labels,
            b_labels,
            c_labels,
            u_labels
        )

        loss = outputs['loss']
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
    print(f"training loss: {total_loss/len(train_loader):.4f}")

    model.eval()
    correct_s = correct_b = correct_c = correct_u = total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids,attention_mask)

            s_pred = torch.argmax(outputs['sentiment_logits'],dim=1)
            b_pred = torch.argmax(outputs['behavior_logits'],dim=1)
            c_pred = torch.argmax(outputs['category_logits'],dim=1)
            u_pred = torch.argmax(outputs['urgency_logits'],dim=1)

            correct_s += (s_pred == batch["sentiment_labels"].to(device)).sum().item()
            correct_b += (b_pred == batch["behavior_labels"].to(device)).sum().item()
            correct_c += (c_pred == batch["category_labels"].to(device)).sum().item()
            correct_u += (u_pred == batch["urgency_labels"].to(device)).sum().item()
            total += input_ids.size(0)
    print(f"Val sentiment accuracy: {correct_s/total:.4f}")
    print(f"Val behavior accuracy: {correct_b/total:.4f}")
    print(f"Val category accuracy: {correct_c/total:.4f}")
    print(f"Val urgency accuracy: {correct_u/total:.4f}")
torch.save(model.state_dict(), "multitask_model.pt")
tokenizer.save_pretrained("multitask_tokenizer")

print("\n✅ Multitask model trained & saved!")

Epoch 1/10
training loss: 1.0682
Val sentiment accuracy: 1.0000
Val behavior accuracy: 0.5090
Val category accuracy: 1.0000
Val urgency accuracy: 0.7980
Epoch 2/10
training loss: 1.0497
Val sentiment accuracy: 1.0000
Val behavior accuracy: 0.4911
Val category accuracy: 1.0000
Val urgency accuracy: 0.7994
Epoch 3/10
training loss: 1.0381
Val sentiment accuracy: 1.0000
Val behavior accuracy: 0.4929
Val category accuracy: 1.0000
Val urgency accuracy: 0.7962
Epoch 4/10
training loss: 1.0299
Val sentiment accuracy: 1.0000
Val behavior accuracy: 0.4859
Val category accuracy: 1.0000
Val urgency accuracy: 0.7973
Epoch 5/10
training loss: 1.0235
Val sentiment accuracy: 1.0000
Val behavior accuracy: 0.5041
Val category accuracy: 1.0000
Val urgency accuracy: 0.7959
Epoch 6/10
training loss: 1.0181
Val sentiment accuracy: 1.0000
Val behavior accuracy: 0.4925
Val category accuracy: 1.0000
Val urgency accuracy: 0.7962
Epoch 7/10
training loss: 1.0109
Val sentiment accuracy: 1.0000
Val behavior accur